In [11]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq


In [12]:
load_dotenv()

# Connect to the Groq API (using the model we know works!)
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7
)

#testing the connection:
#response = llm.invoke("i s astrology a barnum effect only?")


## Making a class for clear instructions with data output type.


In [13]:
class Resume(BaseModel):
    name: str = Field(description="Full name of the candidate")
    email: str | None = Field(description="Get email if present, else none")
    years_exp: float = Field(description="Total years of professional experience")
    skills: list[str] = Field(description="List of technincal skills mentioned")


## Load input data


In [ ]:
# Mention the path to the sample resume file
base_dir = Path.cwd() / "code" / "samples"
sample_path = base_dir / "resume_messy.txt"

if not sample_path.exists():
    base_dir = Path.cwd()
    sample_path = base_dir / "code" / "samples" / "resume_messy.txt"

TEXT = sample_path.read_text(encoding="utf-8")

In [ ]:
# File exists, print the content
print (TEXT)

hey so my name is Aarav Sharma, been working as a backend dev for about 4 and a half
years now. you can reach me at aarav.sharma@example.com or just ping on linkedin.
mostly i do python, fastapi, postgres, some docker and aws. recently started learning
langgraph and building ai agents which is super fun. also know a bit of react but not my
main thing. looking for agentic ai engineer roles now.


## Extract text using LLM without calling the Strutured Class.


In [75]:
# Without structured output, we can just invoke the model and get a response
prompt = f"Form the given resume:\n{TEXT}\n\nExtract the following information:\n- Name\n- Email\n- Years of experience\n- Skills mentioned"
resp_unstructured = llm.invoke(prompt)
resp_unstructured.content


'**Extracted Information**\n\n- **Name:** Aarav Sharma  \n- **Email:** aarav.sharma@example.com  \n- **Years of Experience:** ~4.5\u202fyears (about 4 and a half years)  \n- **Skills Mentioned:**  \n  - Python  \n  - FastAPI  \n  - PostgreSQL  \n  - Docker  \n  - AWS  \n  - LangGraph  \n  - AI agents (agentic AI)  \n  - React (basic knowledge)'

## Extract text using LLM by calling the Strutured Class.

In [80]:
# calling include_raw = True will return the raw response from the model along with the parsed output
resp_structured = llm.with_structured_output(Resume,include_raw=True)
resp_structured

{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FC47A38B90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FC47A39590>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Resume', 'description': '', 'parameters': {'properties': {'name': {'description': 

## Setting up various messages for more precise instructions, like system, user messages.

In [77]:
system ="You are a helpful assistant that extracts structured information from resumes. You will be given a resume text, and your task is to extract the following information: Name, Email, Years of experience, and Skills mentioned. If any of this information is not present in the resume, please return 'None' for that field."
""

message_config =[
    {"role": "system", "content": system},
    {"role": "user", "content":  f"Extract sructured data from this resume: \n {TEXT}"}
]

# Passing the message_config to the structured output model
resp_final = resp_structured.invoke(message_config)
resp_final


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "We need to extract structured data: Name, Email, Years of experience, Skills mentioned. We need to call function. We need to parse the resume text. Name: Aarav Sharma. Email: aarav.sharma@example.com. Years of experience: about 4 and a half years => 4.5 years. Skills: python, fastapi, postgres, docker, aws, langgraph, ai agents, react. We can list skills as strings. Provide function call.\n\nLet's produce JSON object.", 'tool_calls': [{'id': 'fc_4576d072-24bc-460c-8550-9482889544c5', 'function': {'arguments': '{"email":"aarav.sharma@example.com","name":"Aarav Sharma","skills":["python","fastapi","postgres","docker","aws","langgraph","ai agents","react"],"years_exp":4.5}', 'name': 'Resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 322, 'total_tokens': 488, 'completion_time': 0.17275594, 'completion_tokens_details': {'reasoning_tokens': 103}, 'prompt_time': 

## Extracting the final version of our work


In [85]:
resp_final["parsed"]
# Checking the type of the final response: dict
# type(resp_final)


Resume(name='Aarav Sharma', email='aarav.sharma@example.com', years_exp=4.5, skills=['python', 'fastapi', 'postgres', 'docker', 'aws', 'langgraph', 'ai agents', 'react'])